<a href="https://colab.research.google.com/github/induwarakeashitha-droid/yala-maha-assist/blob/main/yalamahatankdataml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Yala-Maha-ml

## load rainfall

In [ ]:
import requests, pandas as pd


ModuleNotFoundError: No module named 'pandas'

In [ ]:

def get_rain(lat, lon, start, end):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "daily": "precipitation_sum",
        "timezone": "Asia/Colombo",
    }
    r = requests.get(url, params=params).json()
    if "daily" not in r:                      # API returned an error
        raise ValueError(r.get("reason", r))
    df = pd.DataFrame({
        "date": pd.to_datetime(r["daily"]["time"]),
        "rain_mm": r["daily"]["precipitation_sum"],
    })
    return df

In [ ]:
lat, lon = 7.364212, 81.021310

rain = get_rain(lat, lon, "2025-01-01", "2025-12-31")
print(rain.shape)
rain.head()

(365, 2)


,date,rain_mm
0,2025-01-01,51.8
1,2025-01-02,2.9
2,2025-01-03,0.4
3,2025-01-04,0.6
4,2025-01-05,0.0


##load tank data

In [ ]:
from google.colab import files
import pandas as pd


In [ ]:
URL = "YOUR_ENDPOINT_URL"      # the link that shows this JSON
r = requests.get(URL, timeout=30)
r.raise_for_status()
js = r.json()

IndexError: list index out of range

In [ ]:
df = pd.json_normalize(js["data"])      # "data" is the list of tanks


In [ ]:
num_cols = ["tot_capacity_mcm", "water_level_msl", "storage_mcm",
            "storage_percentage", "rainfall_last_24h_mm"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")


In [ ]:
df["last_update"] = pd.to_datetime(df["last_update"], utc=True).dt.tz_convert("Asia/Colombo")


In [ ]:

df.to_csv("tanks_snapshot.csv", index=False)
df